# MLB Draft

Two layers: the Baseball Almanac history (1965-2025, names and schools) and the
MLB Stats API detail (2021-2026, with signing bonuses, slot values, school class
and biography). Plus MLB Pipeline's pre-draft prospect rankings.

Covers: `parse_mlb_draft`, `get_drafted_players_mlb`,
`get_drafted_players_all_years_mlb`, `get_drafted_players_college`,
`get_drafted_players_all_years_college`, `print_draft_picks_mlb`,
`print_draft_picks_college`, `draft_pick`, `draft_class`, `draft_history`,
`slot_value`, `signing_bonus`, `bonus_vs_slot`, `overslot_picks`,
`biggest_bonuses`, `draft_demographics`, `conference_draft_counts`,
`state_pipeline`, `prospect_rank`, `prospect_board`, `prospect_vs_actual`,
`biggest_draft_risers`, `biggest_draft_fallers`

In [1]:
from ncaa_bbStats import *

## Draft history, 1965-2025

In [2]:
# Everyone a club drafted in one year
picks = get_drafted_players_mlb("Boston Red Sox", 2025)
print(f"{len(picks)} picks")
print_draft_picks_mlb(picks[:5])

21 picks
2025 Round 1 Pick 15: Kyson Witherspoon - RHP from University of Oklahoma
2025 Round 1 Pick 33: Marcus Phillips - RHP from University of Tennessee
2025 Round 2 Pick 75: Henry Godbout - SS from University of Virginia
2025 Round 3 Pick 87: Anthony Eyanson - RHP from Louisiana State University
2025 Round 4 Pick 118: Mason White - SS from University of Arizona


In [3]:
# Everyone drafted out of one school
picks = get_drafted_players_college("Northeastern", 2025)
print(f"{len(picks)} picks out of Northeastern in 2025")
print_draft_picks_college(picks)

5 picks out of Northeastern in 2025
2025 Round 6 Pick 176: Jordan Gottesman - LHP for San Francisco Giants
2025 Round 7 Pick 206: Cameron Maldonado - OF for San Francisco Giants
2025 Round 13 Pick 399: Jack Goodman - SS for Detroit Tigers
2025 Round 18 Pick 536: Cooper McGrath - RHP for San Francisco Giants
2025 Round 18 Pick 547: Aiven Cabral - RHP for Atlanta Braves


In [4]:
all_picks = get_drafted_players_all_years_college("Northeastern")
print(f"{len(all_picks)} Northeastern players drafted, 1965-2025")

club = get_drafted_players_all_years_mlb("Boston Red Sox")
print(f"{len(club)} players drafted by the Red Sox, all years")

110 Northeastern players drafted, 1965-2025
2134 players drafted by the Red Sox, all years


In [5]:
# parse_mlb_draft scrapes Baseball Almanac live; the cache above is built from it.
# Needs the [scrape] extra and a network connection.
try:
    rows = parse_mlb_draft(2025)
    print(f"scraped {len(rows)} picks; first: {rows[0]}")
except Exception as exc:
    print(f"live scrape skipped ({type(exc).__name__}: {exc})")
    print("The cached data used above needs no network.")

Parsed 615 draft picks for 2025.
scraped 615 picks; first: {'Round': '1', 'Pick': '1', 'Phase': 'JR', 'Player Name': 'Eli Willits', 'Drafted By': 'Washington Nationals', 'POS': 'SS', 'Drafted From': 'Fort Cobb-Broxton High School (Fort Cobb, OK)', 'Year': 2025}


## Draft detail, 2021-2026

Signing bonuses, published slot values, school class, and biography.

In [6]:
import json
print(json.dumps(draft_pick(2024, 1), indent=2))

{
  "age": 23,
  "bats": "L",
  "birth_city": "Hornsby",
  "birth_country": "Australia",
  "birth_date": "2002-08-28",
  "birth_state": "NSW",
  "height": "5' 11\"",
  "is_drafted": true,
  "is_pass": false,
  "mlbam_id": 683953,
  "name": "Travis Bazzana",
  "pick": 1,
  "position": "2B",
  "prospect_rank": 1,
  "round": "1",
  "round_pick": 1,
  "school": "Oregon State",
  "school_class": "4YR JR",
  "school_country": "USA",
  "school_state": "OR",
  "scouting_report_url": "https://www.mlb.com/video/2024-draft-travis-bazzana-2b?t=mlb-draft",
  "signing_bonus": 8950000,
  "slot_value": 10570600,
  "team_id": 114,
  "team_name": "Cleveland Guardians",
  "throws": "R",
  "weight": 199,
  "year": 2024
}


In [7]:
# MLB publishes slot values for the first ten rounds only. Later picks return
# None rather than 0 -- the API literally reports the string "0" there.
print("slot for 2025 pick   1:", f"${slot_value(2025, 1):,}")
print("slot for 2025 pick 100:", f"${slot_value(2025, 100):,}")
print("slot for 2025 pick 400:", slot_value(2025, 400), "(past round 10)")

slot for 2025 pick   1: $11,075,900
slot for 2025 pick 100: $765,400
slot for 2025 pick 400: None (past round 10)


In [8]:
name = "Kade Anderson"
print(f"{name} signed for ${signing_bonus(name, 2025):,}")
print(f"  which is {bonus_vs_slot(name, 2025):.1%} of slot")

Kade Anderson signed for $8,800,000
  which is 92.6% of slot


In [9]:
for pick in draft_class("LSU", 2025):
    bonus = f"${pick['signing_bonus']:,}" if pick["signing_bonus"] else "unsigned"
    print(f"  #{pick['pick']:>3}  round {pick['round']:>4}  "
          f"{pick['name']:24s} {pick['position']:4s} {bonus}")

  #  3  round    1  Kade Anderson            P    $8,800,000
  # 47  round    2  Chase Shores             P    $2,077,200
  # 87  round    3  Anthony Eyanson          P    $1,750,000
  # 95  round    3  Ethan Frey               OF   $997,500
  #185  round    6  Daniel Dickinson         SS   $325,000
  #263  round    9  Jared Jones              1B   $203,600
  #268  round    9  Jacob Mayers             P    $190,000
  #307  round   10  Kade Woods               P    $2,500
  #463  round   15  Conner Ware              P    $150,000


In [10]:
history = draft_history("Vanderbilt", 2021, 2026)
print(f"{len(history)} Vanderbilt players drafted 2021-2026")
first_rounders = [p for p in history if p["round"] == "1"]
print(f"  {len(first_rounders)} first-rounders:",
      [p["name"] for p in first_rounders])

38 Vanderbilt players drafted 2021-2026
  4 first-rounders: ['Jack Leiter', 'Kumar Rocker', 'Spencer Jones', 'Enrique Bradfield Jr.']


In [11]:
print("Biggest 2025 bonuses:")
for pick in biggest_bonuses(2025, n=5):
    print(f"  ${pick['signing_bonus']:>10,}  #{pick['pick']:<4} "
          f"{pick['name']:24s} {pick['school']}")

Biggest 2025 bonuses:
  $ 9,000,000  #4    Ethan Holliday           Stillwater HS
  $ 8,800,000  #3    Kade Anderson            LSU
  $ 8,200,000  #1    Eli Willits              Fort Cobb-Broxton HS
  $ 7,689,525  #2    Tyler Bremner            UC Santa Barbara
  $ 7,250,000  #5    Liam Doyle               Tennessee


In [12]:
print("Signed furthest over slot in 2025:")
for pick in overslot_picks(2025, min_ratio=1.5, n=5):
    print(f"  {pick['bonus_slot_ratio']:.2f}x  #{pick['pick']:<4} "
          f"{pick['name']:24s} ${pick['signing_bonus']:,}")

Signed furthest over slot in 2025:
  5.62x  #181  Josiah Hartshorn         $2,000,000
  5.09x  #127  Briggs McKenzie          $2,997,500
  4.91x  #142  Coy James                $2,500,000
  4.85x  #221  Matthew Fisher           $1,250,000
  3.74x  #253  Camden Lohman            $797,500


In [13]:
demographics = draft_demographics(2025)
print(f"2025 draft: {demographics['picks']} picks")
print(f"  by origin       : {demographics['by_origin']}")
print(f"  mean age        : {demographics['mean_age']}")
print(f"  signed          : {demographics['signed']}")
print(f"  total bonuses   : ${demographics['total_bonus_dollars']:,}")
print(f"  top positions   : {dict(list(demographics['by_position'].items())[:5])}")
print(f"  top states      : {dict(list(demographics['top_states'].items())[:5])}")

2025 draft: 615 picks
  by origin       : {'four_year': 459, 'high_school_or_other': 125, 'junior_college': 31}
  mean age        : 21.26
  signed          : 576
  total bonuses   : $391,247,561
  top positions   : {'P': 358, 'SS': 78, 'OF': 78, 'C': 43, '3B': 21}
  top states      : {'CA': 55, 'FL': 53, 'TX': 51, 'NC': 48, 'GA': 31}


In [14]:
print("Draft picks produced per conference, 2025:")
for row in conference_draft_counts(2025)[:8]:
    print(f"  {row['conference']:16s} {row['picks']:3d} picks   "
          f"${row['bonus_dollars']:>12,}")

Draft picks produced per conference, 2025:


  SEC              107 picks   $  95,003,505
  ACC               61 picks   $  39,604,575
  Big 12            57 picks   $  16,993,576
  Big Ten           34 picks   $  14,937,625
  Sun Belt          21 picks   $   9,443,300
  The American      17 picks   $   4,621,500
  Big West          14 picks   $  10,776,425
  CUSA              14 picks   $   4,528,000


In [15]:
print("Players drafted out of Texas schools:")
for row in state_pipeline("TX"):
    top = row["top_pick"]
    print(f"  {row['season']}  {row['picks']:3d} picks  "
          f"${row['bonus_dollars']:>11,}  best: #{top['pick']} {top['name']}")

Players drafted out of Texas schools:
  2021   60 picks  $ 29,971,900  best: #5 Colton Cowser
  2022   46 picks  $ 19,350,700  best: #12 Jace Jung
  2023   64 picks  $ 25,875,000  best: #8 Blake Mitchell
  2024   40 picks  $ 33,523,800  best: #12 Braden Montgomery
  2025   51 picks  $ 26,869,325  best: #18 Kayson Cunningham
  2026   54 picks  $ 42,375,309  best: #2 Grady Emerson


## Prospect rankings

MLB Pipeline's pre-draft top 250, useful as a benchmark against where players
actually went.

In [16]:
print("Kade Anderson pre-draft rank:", prospect_rank("Kade Anderson", 2025))

print("\n2025 board, top 8:")
for row in prospect_board(2025, n=8):
    origin = "college" if row["is_college"] else "high school"
    print(f"  {row['rank']:>3}. {row['name']:24s} {row['position']:6s} "
          f"{row['school']:24s} ({origin})")

Kade Anderson pre-draft rank: 2

2025 board, top 8:
    1. Ethan Holliday           SS     Stillwater (OK)          (high school)
    2. Kade Anderson            LHP    Louisiana State          (college)
    3. Seth Hernandez           RHP    Corona (CA)              (high school)
    4. Jamie Arnold             LHP    Florida State            (college)
    5. Eli Willits              SS     Fort Cobb-Broxton (OK)   (high school)
    6. Aiva Arquette            SS     Oregon State             (college)
    7. Billy Carlson            SS     Corona (CA)              (high school)
    8. Liam Doyle               LHP    Tennessee                (college)


In [17]:
# Filter the board
print("Top college left-handers, 2025:")
for row in prospect_board(2025, position="LHP", college_only=True, n=5):
    print(f"  {row['rank']:>3}. {row['name']:24s} {row['school']}")

Top college left-handers, 2025:
    2. Kade Anderson            Louisiana State
    4. Jamie Arnold             Florida State
    8. Liam Doyle               Tennessee
   42. Zach Root                Arkansas
   44. Jack Bauer               Mississippi State


In [18]:
comparison = prospect_vs_actual(2025)
print(f"{len(comparison)} players on both the board and the draft record\n")

import statistics
print("rank vs actual pick correlation:",
      round(statistics.correlation([r["prospect_rank"] for r in comparison],
                                   [r["actual_pick"] for r in comparison]), 3))

198 players on both the board and the draft record

rank vs actual pick correlation: 0.546


In [19]:
print("Went much earlier than ranked:")
for row in biggest_draft_risers(2025, n=5):
    print(f"  {row['name']:24s} ranked {row['prospect_rank']:>3} "
          f"-> picked {row['actual_pick']:>3}  ({row['surprise']:+d})")

print("\nSlid furthest:")
for row in biggest_draft_fallers(2025, n=5):
    print(f"  {row['name']:24s} ranked {row['prospect_rank']:>3} "
          f"-> picked {row['actual_pick']:>3}  ({row['surprise']:+d})")

Went much earlier than ranked:
  Michael Oliveto          ranked 219 -> picked  34  (+185)
  Dominick Reid            ranked 209 -> picked  90  (+119)
  Joshua Flores            ranked 243 -> picked 125  (+118)
  Aaron Walton             ranked 161 -> picked  66  (+95)
  Will Hynes               ranked 163 -> picked  70  (+93)

Slid furthest:
  Cam Appenzeller          ranked  58 -> picked 572  (-514)
  Landon Schaefer          ranked 125 -> picked 611  (-486)
  Jacob Parker             ranked 109 -> picked 573  (-464)
  Ethin Bingaman           ranked 150 -> picked 603  (-453)
  Ethan Moore              ranked 100 -> picked 534  (-434)
